In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name == "srcA":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from srcA.data import load_creatives, load_confirmed_personas
from srcA.config import (
    CREATIVES_PATH,
    CONFIRMED_PERSONAS_PATH,
    FUSED_EMBEDDINGS_PATH,
    ARTIFACTS_DIR,
)

print("Project root:", PROJECT_ROOT)

Project root: /Users/vanshikaarora/Desktop/Assignment-VanshikaArora/Memologs-CodeandDocs


In [2]:
import joblib

creatives = load_creatives(CREATIVES_PATH)
confirmed = load_confirmed_personas(CONFIRMED_PERSONAS_PATH)

embeddings = np.load(FUSED_EMBEDDINGS_PATH).astype(np.float32)

classifier = joblib.load(
    ARTIFACTS_DIR / "persona_classifier.pkl"
)

print("Creatives:", len(creatives))
print("Confirmed labels:", len(confirmed))
print("Embeddings:", embeddings.shape)
print("Classifier:", type(classifier.model).__name__)

Creatives: 2882
Confirmed labels: 600
Embeddings: (2882, 512)
Classifier: OneVsRestClassifier


In [3]:
labelled_ids = set(
    confirmed["creative_id"].astype(str)
)

creatives["creative_id"] = creatives["creative_id"].astype(str)

unlabelled_mask = ~creatives["creative_id"].isin(labelled_ids)

unlabelled = creatives[unlabelled_mask].copy()
labelled = creatives[~unlabelled_mask].copy()

print(f"Labelled creatives:   {len(labelled):,}")
print(f"Unlabelled creatives: {len(unlabelled):,}")

Labelled creatives:   600
Unlabelled creatives: 2,282


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

labelled_indices = labelled.index.to_numpy()
unlabelled_indices = unlabelled.index.to_numpy()

labelled_embeddings = embeddings[labelled_indices]
unlabelled_embeddings = embeddings[unlabelled_indices]

# Normalize for cosine similarity
labelled_embeddings = (
    labelled_embeddings /
    np.linalg.norm(
        labelled_embeddings,
        axis=1,
        keepdims=True
    ).clip(min=1e-12)
)

unlabelled_embeddings = (
    unlabelled_embeddings /
    np.linalg.norm(
        unlabelled_embeddings,
        axis=1,
        keepdims=True
    ).clip(min=1e-12)
)

similarities = cosine_similarity(
    unlabelled_embeddings,
    labelled_embeddings
)

nearest_positions = similarities.argmax(axis=1)
top_similarities = similarities.max(axis=1)

nearest_ids = labelled.iloc[
    nearest_positions
]["creative_id"].to_numpy()

audit = unlabelled[
    ["creative_id", "ad_size", "ad_text", "image_path"]
].copy()

audit["top_similarity"] = top_similarities
audit["nearest_labelled_id"] = nearest_ids

In [5]:
probabilities = classifier.model.predict_proba(
    unlabelled_embeddings
)

persona_names = classifier.persona_names

top_indices = probabilities.argmax(axis=1)

audit["top_persona"] = [
    persona_names[i]
    for i in top_indices
]

audit["top_probability"] = probabilities.max(axis=1)

sorted_probabilities = np.sort(
    probabilities,
    axis=1
)

audit["second_probability"] = sorted_probabilities[:, -2]

audit["margin"] = (
    audit["top_probability"]
    - audit["second_probability"]
)

audit.head()

,creative_id,ad_size,ad_text,image_path,top_similarity,nearest_labelled_id,top_persona,top_probability,second_probability,margin
1,c_0001,160x600,CHRYSTAL EVANS HURST\nPRISCILLA SHIRER\nTONY E...,images/c_0001.jpg,0.951008,c_1086,Traveler & Leisure,0.885574,0.064075,0.821499
2,c_0002,728x90,Microsoft\nSeamlessly deliver Al experiences\n...,images/c_0002.jpg,0.773963,c_2470,Tech Enthusiast,0.627584,0.529670,0.097913
5,c_0005,970x90,C\nISTDC Incredible India\nTHE GOLDEN CHARIOT\...,images/c_0005.jpg,0.716849,c_2672,Traveler & Leisure,0.995434,0.111637,0.883797
6,c_0006,728x90,Quest ny\nBuy your own lab tests at Quest.\nPr...,images/c_0006.jpg,0.720733,c_1994,Health & Wellness Seeker,0.983608,0.141181,0.842427
7,c_0007,970x90,Travel\nCST 1016202-80\nCruise U.S. Rivers GET...,images/c_0007.jpg,0.772929,c_1278,Traveler & Leisure,0.663938,0.168733,0.495206


In [7]:
def current_decision(row):
    if (
        row["top_similarity"] >= 0.85
        and row["top_probability"] >= 0.75
    ):
        return "suggest"

    if (
        row["top_similarity"] >= 0.70
        and row["top_probability"] >= 0.55
    ):
        return "suggest_with_caution"

    return "abstain"


audit["decision"] = audit.apply(
    current_decision,
    axis=1
)

In [8]:
weak_suggestions = audit[
    (audit["decision"] == "suggest") &
    (audit["top_similarity"] < 0.60)
].sort_values(
    ["top_similarity", "top_probability"]
)

print(
    f"Strong LR suggestions with top similarity < 0.60: "
    f"{len(weak_suggestions)}"
)

display(
    weak_suggestions[
        [
            "creative_id",
            "top_persona",
            "top_probability",
            "top_similarity",
            "margin",
            "nearest_labelled_id",
            "ad_text",
        ]
    ].head(30)
)

Strong LR suggestions with top similarity < 0.60: 0


,creative_id,top_persona,top_probability,top_similarity,margin,nearest_labelled_id,ad_text


In [9]:
weak_suggestions = weak_suggestions.head(20)

for _, row in weak_suggestions.iterrows():

    print("=" * 80)
    print("Creative:", row["creative_id"])
    print("Prediction:", row["top_persona"])
    print("Probability:", round(row["top_probability"], 3))
    print("Nearest labelled:", row["nearest_labelled_id"])
    print("Similarity:", round(row["top_similarity"], 3))
    print("Margin:", round(row["margin"], 3))
    print("Text:", row["ad_text"])

    image_path = PROJECT_ROOT / row["image_path"]

    if image_path.exists():
        image = plt.imread(image_path)

        plt.figure(figsize=(8, 4))
        plt.imshow(image)
        plt.axis("off")
        plt.show()

In [11]:
# Improved evidence-aware abstention thresholds

STRONG_SIMILARITY = 0.60
MEDIUM_SIMILARITY = 0.45

STRONG_PROBABILITY = 0.75
MEDIUM_PROBABILITY = 0.55

STRONG_MARGIN = 0.15

print("Strong similarity:", STRONG_SIMILARITY)
print("Medium similarity:", MEDIUM_SIMILARITY)
print("Strong probability:", STRONG_PROBABILITY)
print("Medium probability:", MEDIUM_PROBABILITY)
print("Strong margin:", STRONG_MARGIN)

Strong similarity: 0.6
Medium similarity: 0.45
Strong probability: 0.75
Medium probability: 0.55
Strong margin: 0.15


In [12]:
def improved_decision(row):

    similarity = row["top_similarity"]
    probability = row["top_probability"]
    margin = row["margin"]

    # Strong evidence
    if (
        similarity >= STRONG_SIMILARITY
        and probability >= STRONG_PROBABILITY
        and margin >= STRONG_MARGIN
    ):
        return "suggest"

    # Moderate evidence
    if (
        similarity >= MEDIUM_SIMILARITY
        and probability >= MEDIUM_PROBABILITY
    ):
        return "suggest_with_caution"

    # Weak/no evidence
    return "abstain"


audit["improved_decision"] = audit.apply(
    improved_decision,
    axis=1
)

In [13]:
audit["improved_decision"].value_counts()

improved_decision
suggest                 1214
suggest_with_caution     611
abstain                  457
Name: count, dtype: int64

In [14]:
display(
    audit[
        [
            "creative_id",
            "top_persona",
            "top_probability",
            "top_similarity",
            "margin",
            "decision",
            "improved_decision",
            "nearest_labelled_id",
        ]
    ].sort_values(
        "top_similarity"
    ).head(30)
)

,creative_id,top_persona,top_probability,top_similarity,margin,decision,improved_decision,nearest_labelled_id
2257,c_2257,Traveler & Leisure,0.588829,0.449290,0.484490,abstain,abstain,c_0345
2445,c_2445,Traveler & Leisure,0.908574,0.465948,0.741993,abstain,suggest_with_caution,c_1203
1711,c_1711,Small Business & B2B,0.539412,0.467601,0.130494,abstain,abstain,c_1902
891,c_0891,Tech Enthusiast,0.443667,0.484775,0.196536,abstain,abstain,c_0507
2206,c_2206,Traveler & Leisure,0.871837,0.505406,0.462022,abstain,suggest_with_caution,c_2083
1658,c_1658,Small Business & B2B,0.755469,0.512832,0.404024,abstain,suggest_with_caution,c_2801
634,c_0634,Traveler & Leisure,0.845994,0.524897,0.485356,abstain,suggest_with_caution,c_1061
811,c_0811,Traveler & Leisure,0.375235,0.526102,0.235702,abstain,abstain,c_2467
2635,c_2635,Traveler & Leisure,0.952722,0.526266,0.840803,abstain,suggest_with_caution,c_2437
1772,c_1772,Traveler & Leisure,0.951553,0.526834,0.795357,abstain,suggest_with_caution,c_0559


In [15]:
audit["improved_decision"].value_counts()

improved_decision
suggest                 1214
suggest_with_caution     611
abstain                  457
Name: count, dtype: int64